# N-Ways to Perform an RF Replay Attack

**Hello!** In this workshop, you’ll learn how to replay RF signals using SDR! 

Although this project is somewhat basic, it does **step-by-step** show you the process of transforming radio signals into data bits, recording them and replaying them to do a replay attack. 

We also want you to have fun! Each step in this **notebook** includes starter project, and we’ll also provide solutions if you need them. 

Let’s get started!

## Learning Objectives
In this module, you will learn about the following:
- how replay attacks work
- saving waveform to a file
- using the **inspectrum** software for signal analysis
- On-Off-Keying (OOK)
- transmitting using a transmit-capable SDR, **HackRF**
- replaying a captured radio signal
- synthesizing signal from bits
- analysis using **rtl-433** and **Flipper Zero**
- using **Universal Radio Hacker (URH)** tool
- using **GNU Radio** in every step of replay attack

## Prerequisites
- general Python knowledge
- general purpose SDR tools or GNU Radio, installed locally.
- RTL-SDR to receive and transmit-capable SDR to transmit RF signals  

# I. Replay Attacks
Replay attacks are a fundamental technique in wireless security testing, where **a previously captured RF signal is retransmitted to trigger the same action**—like unlocking a door or turning on a device. In this demonstration, I explore several different tools and workflows that can be used to carry out a replay attack, each with varying levels of complexity and control. From user-friendly devices like the Flipper Zero to more advanced signal processing platforms like GNU Radio, this hands-on approach highlights the strengths and limitations of each method.

There are many ways for replay attacks, and the methods can be chosen based upon your **needs and the cost, complexity, and versatility** of the devices and tools available.

## Setup
In this module we will be using a simple RF radio device, a **remote control** and **HackRF** in order to receive and transmit signals. The remote controller we are going to be copying has 2 buttons: A green button for ringing the green keyring and a red button for ringing the red keyring. Simple enough. We are going to work on the green button for this project. We will first supply signal from original transmitter, capture it, and see if we can replay it later. If you have issues capturing a file, or simply you don't have any capturing device with you, you can copy the pre-captured files from the solutions folder into your working directory. There exist `kf_blue.cf32`, and `kf_red.cf32` pre-captured files in the solutions folder. 

In the sections that follow, we'll first cover what I call a **simple** replay attack. This does not involve analyzing signal what it contains or what is its modulation scheme. We will simply record and replay. We can do this in several ways. In the first part we will use HackRF along with **GNU Radio**, in the second part HackRF tool **hackrf_transfer** will be used. Then we'll use **Flipper Zero** device and popular tools like **inspectrum, rtl_433, URH**. 

Lastly, in section VIII , RF data from the remote control will be analyzed and decoded, bits will be extracted and then we will do replay attack with synthesized signal. All of these can be done with **GNU Radio!** This may be somewhat more difficult to do but there are many good open tools for this purpose. 

| **Workshop** | **Setup** |
|:---:|:---:|
| <img src="nbimages/setup_hackrf.jpg" alt="Drawing" /> | <img src="nbimages/setup_hightek.png" alt="Drawing" /> |
| <img src="nbimages/setup_led.png" alt="Drawing" /> | <img src="nbimages/setup_workshop.jpg" alt="Drawing" /> |


# II. Simple Replay Attack with GNU Radio
HackRF can be used in conjunction with GNU Radio flow-graphs to capture, save, and replay RF signals. This method requires some prior knowledge on GNU Radio. But the steps we will follow are: 
1. **Search** the signal of interest  
2. **Save** the waveform ie, capture
3. **Make sure** it has been captured correctly
4. **Replay** the captured signal

## 1.Searching the Signal
The first thing to do is to try to find the exact **operating frequency** of the device. You can sweep RF spectrum with wide RF bandwidths. HackRF supports **scanning wide RF bandwidths**, making it an essential tool for **signal discovery**. With **a maximum sample rate of 20 MS/s**, HackRF can capture up to 20 MHz of instantaneous bandwidth. This is critical when searching for unknown or short-duration signals, as narrowband scanning can easily miss transient transmissions. By capturing a wide swath of spectrum at once, HackRF allows you to monitor multiple channels and quickly identify signal activity across a large frequency range. 

<img src="nbimages/grc_1_monitor_wide.png" alt="Drawing" />

So, it is a good idea first to look at **ISM bands** for this task involving a remote. ISM stands for **Industrial Scientific and Medical**. You would use such a band because it's unlicensed in that you are allowed to use equipment that operates in that band without you having a license yourself. There may be rules however about the authorization of the equipment even though there aren't rules about the authorization of the user of the equipment so we call those unlicensed bands. 

<img src="nbimages/ISM_bands.png" alt="Drawing" />

So, we will start by making a flow graph to receive a signal from the remote control and will start with just visualize the signal we have recived. GNU Radio flowgraph sketched below helps you to monitor spectrum for signal of interest. 

<img src="nbimages/grc_1_monitor.png" alt="Drawing" />

### Hint: Running GNU Radio from Terminal
You can run a GNU Radio flow graph right from the **terminal**. 
1. You need to compile the flowgraph, 
2. Then Run the python script generated

Note: Do not forget to specify the filename as parameter if there is. 

There is a pre-made flowgraph, `Monitor.grc` in the solutions directory. 

Click the `...` to run the solution. 

In [4]:

!grcc solutions/1_Monitor.grc 
!/usr/bin/python3 Monitor.py --frequency=433e6

bb60_iq.block.yml                        block: error: Value type 'NoneType' for 'inputs' not in valid types
sm_iq.block.yml                          block: error: Value type 'NoneType' for 'inputs' not in valid types
>>> Warning: vocoder_codec2_decode_ps - option_attributes are for enums only, ignoring
>>> Warning: vocoder_codec2_encode_sp - option_attributes are for enums only, ignoring
vsg60_iqin.block.yml                     block: error: Value type 'NoneType' for 'outputs' not in valid types
<<< Welcome to GNU Radio Companion Compiler 3.10.4.0 >>>

Block paths:
	/home/murat/.grc_gnuradio
	/usr/local/share/gnuradio/grc/blocks

>>> Loading: /home/murat/gitdir/EUGRDays2026/solutions/1_Monitor.grc
>>> Generating: /home/murat/gitdir/EUGRDays2026/Monitor.py
[INFO] Opening HackRF One #0 66a062dc2551639f...
[+] Available device: 0:0 <NVIDIA Corporation> NVIDIA GeForce RTX 4080 Laptop GPU
[+] Available device: 1:0 <GenuineIntel> pthread-13th Gen Intel(R) Core(TM) i7-13700HX
[+] Selected de

**Note 1**: You will see a **continous peak at the center**, i.e., DC spike. It is not a real signal but just an **artifact** from HackRF.  

**Note 2**: You might observe a **mirror image** of the signal if you set gain too high! 

## 2. Saving the Waveform to a File

After adjusting the center frequency for optimum gain and clarity, we can use the following flowgraph `Capture.grc` and then **press** the peripheral remote button and ultimately **capture** the RF signal burst to a file. This is just a raw digital waveform. It's the information coming from the Hack RF being saved directly to a file. We are not demodulating, filtering, or doing any kind of processing whatsoever. We are just taking the data from our SDR, Hack RF, and sending it to a file. 

<img src="nbimages/grc_2_capture.png" alt="Drawing" />

There is a pre-made flowgraph, `2_Capture.grc` in the solutions directory. 

Click the `...` to run the solution. 

In [5]:

!grcc solutions/2_Capture.grc 
!/usr/bin/python3 Capture.py --filename='keyfinder_red.dat' --frequency=433e6

bb60_iq.block.yml                        block: error: Value type 'NoneType' for 'inputs' not in valid types
sm_iq.block.yml                          block: error: Value type 'NoneType' for 'inputs' not in valid types
>>> Warning: vocoder_codec2_decode_ps - option_attributes are for enums only, ignoring
>>> Warning: vocoder_codec2_encode_sp - option_attributes are for enums only, ignoring
vsg60_iqin.block.yml                     block: error: Value type 'NoneType' for 'outputs' not in valid types
<<< Welcome to GNU Radio Companion Compiler 3.10.4.0 >>>

Block paths:
	/home/murat/.grc_gnuradio
	/usr/local/share/gnuradio/grc/blocks

>>> Loading: /home/murat/gitdir/EUGRDays2026/solutions/2_Capture.grc
>>> Generating: /home/murat/gitdir/EUGRDays2026/Capture.py
[INFO] Opening HackRF One #0 66a062dc2551639f...
[+] Available device: 0:0 <NVIDIA Corporation> NVIDIA GeForce RTX 4080 Laptop GPU
[+] Available device: 1:0 <GenuineIntel> pthread-13th Gen Intel(R) Core(TM) i7-13700HX
[+] Selected de

## 3. Checking the File
Make sure your record actually contains something meaningful! In order to **verify** that we have captured our signal of interest we will visualize things. We will again use a QT GUI Frequency Sink however we probably want to slow it down with a throttle block in order to perform in the same speed as real time. We should set the sample rate to be the same sample rate that we have used for the capture which was 2 million samples per second, 2 MSPS. We will give this flow graph a name `CheckRecord.grc`. Run the GR flowgraph below to see what the capture contains. 

<img src="nbimages/grc_3_check.png" alt="Drawing" />

There is a pre-made flowgraph, `3_Check.grc` in the solutions directory. 

Click the `...` to run the solution. 

In [6]:

!grcc solutions/3_Check.grc 
!/usr/bin/python3 Check.py --filename='keyfinder_red.dat' --frequency=433e6

bb60_iq.block.yml                        block: error: Value type 'NoneType' for 'inputs' not in valid types
sm_iq.block.yml                          block: error: Value type 'NoneType' for 'inputs' not in valid types
>>> Warning: vocoder_codec2_decode_ps - option_attributes are for enums only, ignoring
>>> Warning: vocoder_codec2_encode_sp - option_attributes are for enums only, ignoring
vsg60_iqin.block.yml                     block: error: Value type 'NoneType' for 'outputs' not in valid types
<<< Welcome to GNU Radio Companion Compiler 3.10.4.0 >>>

Block paths:
	/home/murat/.grc_gnuradio
	/usr/local/share/gnuradio/grc/blocks

>>> Loading: /home/murat/gitdir/EUGRDays2026/solutions/3_Check.grc
>>> Generating: /home/murat/gitdir/EUGRDays2026/Check.py
[+] Available device: 0:0 <NVIDIA Corporation> NVIDIA GeForce RTX 4080 Laptop GPU
[+] Available device: 1:0 <GenuineIntel> pthread-13th Gen Intel(R) Core(TM) i7-13700HX
[+] Selected device: NVIDIA GeForce RTX 4080 Laptop GPU
[w] CL Error

## 4. Replaying a Captured Radio Signal
HackRF can **replay** (transmit) the raw RF signal in the saved file and thereby invoke the desired peripheral activity without the use of the physical remote. 

<img src="nbimages/grc_4_replay_simple.png" alt="Drawing" />

There is a pre-made flowgraph, `4_Replay.grc` in the solutions directory that simply transmits the recorded signal.  

Click the `...` to run the solution. 

In [7]:

!grcc solutions/4_Replay.grc 
!/usr/bin/python3 Replay.py --filename='keyfinder_red.dat' --frequency=433e6

bb60_iq.block.yml                        block: error: Value type 'NoneType' for 'inputs' not in valid types
sm_iq.block.yml                          block: error: Value type 'NoneType' for 'inputs' not in valid types
>>> Warning: vocoder_codec2_decode_ps - option_attributes are for enums only, ignoring
>>> Warning: vocoder_codec2_encode_sp - option_attributes are for enums only, ignoring
vsg60_iqin.block.yml                     block: error: Value type 'NoneType' for 'outputs' not in valid types
<<< Welcome to GNU Radio Companion Compiler 3.10.4.0 >>>

Block paths:
	/home/murat/.grc_gnuradio
	/usr/local/share/gnuradio/grc/blocks

>>> Loading: /home/murat/gitdir/EUGRDays2026/solutions/4_Replay.grc
>>> Generating: /home/murat/gitdir/EUGRDays2026/Replay.py
[INFO] Opening HackRF One #0 66a062dc2551639f...
[+] Available device: 0:0 <NVIDIA Corporation> NVIDIA GeForce RTX 4080 Laptop GPU
[+] Available device: 1:0 <GenuineIntel> pthread-13th Gen Intel(R) Core(TM) i7-13700HX
[+] Selected devi

## 5. Replaying better 
We can do much better if we apply some **gain** and **filtering** before transmission. 

Amplifying our signal digitally allows us to use larger portion of the dynamic range. This is done with the multiplication block. 

After amplification, we apply bandpass filter, being a much better neighbor on the spectrum. 

<img src="nbimages/grc_5_replay_better.png" alt="Drawing" />

Pre-made flowgraph `ReplayBetter.grc` does all these before sending. 

Click the `...` to run the solution. 

In [8]:

!grcc solutions/5_ReplayBetter.grc 
!/usr/bin/python3 ReplayBetter.py --filename='keyfinder_red.dat' --frequency=433e6

bb60_iq.block.yml                        block: error: Value type 'NoneType' for 'inputs' not in valid types
sm_iq.block.yml                          block: error: Value type 'NoneType' for 'inputs' not in valid types
>>> Warning: vocoder_codec2_decode_ps - option_attributes are for enums only, ignoring
>>> Warning: vocoder_codec2_encode_sp - option_attributes are for enums only, ignoring
vsg60_iqin.block.yml                     block: error: Value type 'NoneType' for 'outputs' not in valid types
<<< Welcome to GNU Radio Companion Compiler 3.10.4.0 >>>

Block paths:
	/home/murat/.grc_gnuradio
	/usr/local/share/gnuradio/grc/blocks

>>> Loading: /home/murat/gitdir/EUGRDays2026/solutions/5_ReplayBetter.grc
>>> Generating: /home/murat/gitdir/EUGRDays2026/ReplayBetter.py
[INFO] Opening HackRF One #0 66a062dc2551639f...
[+] Available device: 0:0 <NVIDIA Corporation> NVIDIA GeForce RTX 4080 Laptop GPU
[+] Available device: 1:0 <GenuineIntel> pthread-13th Gen Intel(R) Core(TM) i7-13700HX
[+] S

# III. Simple Replay Attack with HackRF Tools
One of the quickest ways to replay an RF signal when the signal center frequency is known is using the HackRF **hackrf_transfer** utility. You can use the following command to capture a radio signal and save it to a file with hackrf_transfer. 

In [23]:
!hackrf_transfer -s 2000000 -f 433000000 -n 20000000 -r hackrf_transfer_433M_2MHz.cs8 -a 1 -g 32

call hackrf_set_sample_rate(2000000 Hz/2.000 MHz)
call hackrf_set_hw_sync_mode(0)
call hackrf_set_freq(433000000 Hz/433.000 MHz)
call hackrf_set_amp_enable(1)
samples_to_xfer 20000000/20Mio
Stop with Ctrl-C
 3.9 MiB / 1.000 sec =  3.9 MiB/second, average power -39.8 dBfs
 3.9 MiB / 1.000 sec =  3.9 MiB/second, average power -16.6 dBfs
 3.9 MiB / 1.000 sec =  3.9 MiB/second, average power -3.9 dBfs
 3.9 MiB / 1.000 sec =  3.9 MiB/second, average power -4.3 dBfs
 4.2 MiB / 1.000 sec =  4.2 MiB/second, average power -4.4 dBfs
 3.9 MiB / 1.000 sec =  3.9 MiB/second, average power -4.3 dBfs
 3.9 MiB / 1.000 sec =  3.9 MiB/second, average power -7.6 dBfs
 4.2 MiB / 1.000 sec =  4.2 MiB/second, average power -40.3 dBfs
 3.9 MiB / 1.000 sec =  3.9 MiB/second, average power -40.1 dBfs
 3.9 MiB / 1.000 sec =  3.9 MiB/second, average power -40.1 dBfs
 0.3 MiB / 0.030 sec =  8.7 MiB/second, average power -40.2 dBfs

Exiting...
Total time: 10.03032 s
hackrf_stop_rx() done
hackrf_close() done
hackrf

Having saved the signal, you can transmit it with the following command. 

<mark>Keep in mind that hackrf_transfer utility uses signed 8 bit format by default. So if you just play the GNU Radio record files, it will not work! That's because GNU Radio uses floating point format. </mark>


In [24]:
!hackrf_transfer -s 2000000 -f 433000000 -R -t hackrf_transfer_433M_2MHz.cs8 -a 1 -x 32

call hackrf_set_sample_rate(2000000 Hz/2.000 MHz)
call hackrf_set_hw_sync_mode(0)
call hackrf_set_freq(433000000 Hz/433.000 MHz)
call hackrf_set_amp_enable(1)
Stop with Ctrl-C
 3.9 MiB / 1.000 sec =  3.9 MiB/second, average power -39.8 dBfs
 3.9 MiB / 1.000 sec =  3.9 MiB/second, average power -16.6 dBfs
 3.9 MiB / 1.000 sec =  3.9 MiB/second, average power -3.9 dBfs
 4.2 MiB / 1.000 sec =  4.2 MiB/second, average power -4.3 dBfs
 3.9 MiB / 1.000 sec =  3.9 MiB/second, average power -4.4 dBfs
 3.9 MiB / 1.000 sec =  3.9 MiB/second, average power -4.3 dBfs
 3.9 MiB / 1.000 sec =  3.9 MiB/second, average power -7.6 dBfs
 4.2 MiB / 1.000 sec =  4.2 MiB/second, average power -40.3 dBfs
 3.9 MiB / 1.000 sec =  3.9 MiB/second, average power -40.1 dBfs
 3.9 MiB / 1.000 sec =  3.9 MiB/second, average power -40.1 dBfs
 3.9 MiB / 1.000 sec =  3.9 MiB/second, average power -39.8 dBfs
 4.2 MiB / 1.000 sec =  4.2 MiB/second, average power -14.8 dBfs
 3.9 MiB / 1.000 sec =  3.9 MiB/second, average p

# IV. Simple Replay Attack with Flipper Zero
The Flipper Zero is a portable, open-source multi-tool designed for pentesters, hardware enthusiasts, and hackers. It comes equipped with a range of built-in radios and interfaces, including **RFID, infrared, GPIO, NFC, iButton, and sub-GHz RF**. One of its most popular features is its sub-GHz module, which allows users to interact with a variety of RF-based devices such as garage doors, car key fobs, and remote switches.

To capture and replay an RF signal using Flipper Zero, first open **Flipper Lab**, web-based platform for the Flipper device. 
<img src="nbimages/Flipper_1_Lab.png" alt="Drawing" />

Navigate to the **Sub-GHz menu** and select **Read** mode. 
|**Sub GHz**|**Read Menu**|
|:---:|:---:|
| <img src="nbimages/Flipper_2_Sub_GHz.png" alt="Drawing" /> | <img src="nbimages/Flipper_3_Read.png" alt="Drawing" /> |

Select the **configuration** like below: 
|**Configuration**|**Scanning...**|
|:---:|:---:|
| <img src="nbimages/Flipper_4_Config.png" alt="Drawing" /> | <img src="nbimages/Flipper_5_Scanning.png" alt="Drawing" /> |

**Press** the button on the original remote near the Flipper, and it will display the frequency and modulation used. 

Once the signal is captured and saved, you can **replay** it by selecting the saved file and choosing **Send**. This feature is particularly useful for security testing of poorly secured or unencrypted RF systems. 
|**Capture**|**Details**|
|:---:|:---:|
| <img src="nbimages/Flipper_6_KeyRedPress.png" alt="Drawing" /> | <img src="nbimages/Flipper_7_Details.png" alt="Drawing" /> |

Note: Before leaving this section take a note of keys Flipper has obtained. See the cell below for the solution!


* Key red:   0x00 87 A8 88
* Key green: 

# V. Easy Decoding with rtl_433
**rtl_433** is a versatile open-source tool used with inexpensive USB software-defined radio (SDR) receivers, such as the RTL-SDR dongle. It is designed to decode a wide range of radio transmissions in the 300–928 MHz ISM bands, including signals from weather stations, tire pressure monitors, remote controls, and various IoT sensors. The tool is widely used by hobbyists, researchers, and developers to analyze and collect data from RF devices operating on common unlicensed frequencies.

To capture and analyze a signal from a basic RF remote using rtl_433, connect your SDR dongle to your machine and launch the rtl_433 command-line tool. When the remote button is pressed near the SDR antenna, rtl_433 attempts to demodulate and decode the signal. If the protocol is recognized, decoded information such as device ID and button press will be printed to the terminal. While rtl_433 is primarily for receiving and decoding signals, not replaying them, the captured data can be saved and further analyzed to understand the protocol. Replay functionality would require a separate SDR transmitter and additional tools like GNU Radio or Universal Radio Hacker (URH).

**Luckily, rtl_433 also works with HackRF**. Let's see what information it will give us. 

In [2]:
!rtl_433 -d "soapy=0,driver=hackrf" -f 433.8e6 -s 2e6

rtl_433 version 23.11-133-g321f231c branch master at 202406140956 inputs file rtl_tcp RTL-SDR SoapySDR with TLS
Using device HackRF One: clock source=internal part id=a000cb3c006f4358 serial=000000000000000066a062dc2551639f version=2024.02.1 
Found 1 antenna(s): TX/RX 
Found 3 gain(s): LNA 0 - 116 (step 0) AMP 0 - 116 (step 0) VGA 0 - 116 (step 0) 
Found 1 frequencies: RF 
Found 1 frequency range(s): 0 - 7250000000 (step 0) 
Found 20 sample rate range(s): 1000000 2000000 3000000 4000000 5000000 6000000 7000000 8000000 9000000 10000000 11000000 12000000 13000000 14000000 15000000 16000000 17000000 18000000 19000000 20000000 
Found 16 bandwidth range(s): 1750000 - 1750000 (step 0) 2500000 - 2500000 (step 0) 3500000 - 3500000 (step 0) 5000000 - 5000000 (step 0) 5500000 - 5500000 (step 0) 6000000 - 6000000 (step 0) 7000000 - 7000000 (step 0) 8000000 - 8000000 (step 0) 9000000 - 9000000 (step 0) 10000000 - 10000000 (step 0) 12000000 - 12000000 (step 0) 14000000 - 14000000 (step 0) 15000000 

It should produce an output like below: 
```
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _
time      : 2025-08-01 18:38:47
model     : Akhan-100F14 ID (20bit): 0x4b945
Data (4bit): 0x2 (Unlock)
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _
```

This seems consistent with the keys we have found earlier with Flipper.  

# VI. Decoding with Inspectrum
In order to take a look at what is in this waveform a little bit more in more detail we will use a piece of software called **Inspectrum**.

## 1. Spectrogram View
Inspectrum provides us with spectrogram which is a two-dimensional picture of the information that is saved in the capture
file. What we have is **frequency** on the vertical axis and **time** on the horizontal axis. 

We can open Inspectrum from terminal by specifying sampling rate and file name like below. 

In [9]:
!inspectrum -r 2e6 keyfinder_red.dat

I you scroll right a little bit until, we see things happening a little bit above our center frequency. We can change things like the FFT size which which affects the height of the analysis and we can change the zoom which affects the width. We can also change the color scheme depending on basically setting thresholds of which color are applied to which colors are applied to which power levels. After adjusting things like, you can see that the capture contains 6 bursts of data. The frequency is nearly consistent because everything falls in a horizontal line but the power level goes on and off very dramatically. You can see this is the
beginning of **On-Off-Keying (OOK)** that's pretty cool that we can see that just by looking at the waveform with this tool. 

<img src="nbimages/inspectrum_1_bursts.png" alt="Drawing" />

## 2. Amplitude Plot
Let's take a closer look at one of the bursts. Zooming in on time axis, we can see clearly that  it transmits for a while and then it's just back to noise and then it transmits for a while and then we're back to noise again. Let's take a look at how we can demodulate this and the nice thing is that Spectrum actually has some built-in demodulation capability. You have to kind of do a lot of things by hand but you can demodulate a signal like this using nothing but in Spectrum. So, we know that the amplitude is being modulated here the transmitter is basically being switched from full amplitude to zero amplitude that is completely off so what we need to do is add an **amplitude plot**. The the red line in the center is the center frequency of my tuning and the white lines at the top and bottom are showing the edges of the filter that I'm using to filter this channel and so you can kind of adjust if you want a wider filter you can drag them out and if you want a narrower filter. 

<img src="nbimages/inspectrum_2_amplitude_plot.png" alt="Drawing" />

## 3. Thresholding
The next thing that we can do is you can take this amplitude demodulated signal and add another derived plot, **threshold** plot, and now anything below this dotted line will be counted as a zero and anything above the dotted line will be counted as a one. So, now we have a nice clean representation of kind of all the ons and offs in this signal. 

<img src="nbimages/inspectrum_3_thresholding.png" alt="Drawing" /> 

## 4. Measuring the baudrate
The next thing we need to do is figure out when we're going to measure this signal to see you know where are the ones and zeros and to do that in Spectrum you need to turn on the **cursors**. Cursors have a bunch of uses you can also use them just to measure the baud rates of signals. 

<img src="nbimages/inspectrum_4_measuring_baud.png" alt="Drawing" /> 

## 5. Obtaining Bits
We can start by using cursor for one bit period but eventually once you have the cursor set up properly you can also use them to demodulate and extract data from things. Just looking at kind of one bit period where the transmitter is on you can get a rough idea how long burst will be. Then gradually increase symbol count to 10, 50, and finally to 100. Make sure that we're staying lined up with the bit periods there like my dotted lines are lining up exactly with the bit transitions. So, once you have everything lined up and we're lined up on the threshold plot you can right click there and say **extract symbols**. You can either go to standard output or to the  clipboard. If you choose clipboard, clipboard should have a set of bits okay one zero one zero one zero that sounds about right for the start of the signal and then eventually we get into where the actual bits of the message are. 

<img src="nbimages/inspectrum_5_bits.png" alt="Drawing" /> 

We will use an online web application called **CyberChef** to understand what those bits are. 
- Open [CyberChef](https://gchq.github.io/CyberChef/) 
- Add these operations: **Split**, **From Binary**, **To Hex**
It will display a hexadecimal output like this: **e8 88 8e ee e8 e8 e8 88 e8 88 e8 88 04**
<img src="nbimages/CyberChef_1.png" alt="Drawing" /> 
You can use the following link for this. 

[CyberChefRecipe](https://gchq.github.io/CyberChef/#recipe=Split(',','%5C%5Cn')From_Binary('Space',8)To_Hex('Space',0))

This hexadecimal is quite different from what we have seen before. Interestingly, there are only two hex literals 8 and e. Investigating this a little bit, if we assume **1, 0, 0, 0** bit sequence corresponds to **0**, and **1, 1, 1, 0** to **1**, we again obtain **0x87A888**. All you need, is to add **Find/Replace** operations to the previous recipe. See the full CyberChef recipe below. 
<img src="nbimages/CyberChef_2.png" alt="Drawing" /> 

[Full recipe](https://gchq.github.io/CyberChef/#recipe=Find_/_Replace(%7B'option':'Regex','string':'1,%201,%201,%200'%7D,'1',true,false,true,false)Find_/_Replace(%7B'option':'Regex','string':'1,%200,%200,%200'%7D,'0',true,false,true,false)Split(',','')From_Binary('Space',8)To_Hex('Space',0)&input=MSwgMSwgMSwgMCwgMSwgMCwgMCwgMCwgMSwgMCwgMCwgMCwgMSwgMCwgMCwgMCwgMSwgMCwgMCwgMCwgMSwgMSwgMSwgMCwgMSwgMSwgMSwgMCwgMSwgMSwgMSwgMCwgMSwgMSwgMSwgMCwgMSwgMCwgMCwgMCwgMSwgMSwgMSwgMCwgMSwgMCwgMCwgMCwgMSwgMSwgMSwgMCwgMSwgMCwgMCwgMCwgMSwgMCwgMCwgMCwgMSwgMCwgMCwgMCwgMSwgMSwgMSwgMCwgMSwgMCwgMCwgMCwgMSwgMCwgMCwgMCwgMSwgMCwgMCwgMCwgMSwgMSwgMSwgMCwgMSwgMCwgMCwgMCwgMSwgMCwgMCwgMCwgMSwgMCwgMCwgMCwgMCwgMSwgMCwgMCwg)

You can also use the following Python script to obtain hex key. 

In [1]:
import bitarray
# read data from file
openFile = open("solutions/bits_red.txt", "r")
str = openFile.read().rstrip()
openFile.close()
data= [int(e) if e.strip().isdigit() else e for e in str.split(',')]
data = data[:-1]
print(bitarray.bitarray(data).tobytes())
print (''.join('{:02x}'.format(x) for x in bitarray.bitarray(data).tobytes()))

b'\xe8\x88\x8e\xee\xe8\xe8\xe8\x88\xe8\x88\xe8\x88@'
e8888eeee8e8e888e888e88840


# VII. Universal Radio Hacker (URH)
**Universal Radio Hacker (URH)** is another powerful open-source application for analyzing unknown wireless protocols. It provides a user-friendly interface for capturing, demodulating, visualizing, and interpreting raw RF signals. URH is commonly used by security researchers and radio enthusiasts to reverse-engineer wireless communication from devices such as remote controls, sensors, and key fobs. It supports a wide range of SDR hardware like HackRF, RTL-SDR, and USRP. Visit application's GitHub page for more information. [https://github.com/jopohl/urh](URH)

<mark>As written in the user guide file ending determines how URH handles the signal. URH  understands these file endings:
- .complex files with complex64 samples (32 Bit float for I and Q, respectively). This is
the default signal file format and will also be used in case the file has no ending at all.
- .complex16u using two unsigned 8 Bit integers for I and Q
- .complex16s using two signed 8 Bit integers for I and Q</mark>

In [3]:
!urh

[INFO::settings.py::get_receive_buffer_size] Allocate receive buffer with 16,295GB
[INFO::Device.py::start_rx_mode] HackRF: Starting RX Mode
[INFO::Device.py::log_retcode] HackRF-SETUP: Success
[INFO::Device.py::log_retcode] HackRF-SET_FREQUENCY to 433.5M: Success
[INFO::Device.py::log_retcode] HackRF-SET_SAMPLE_RATE to 2M: Success
[INFO::Device.py::log_retcode] HackRF-SET_BANDWIDTH to 2M: Success
[INFO::Device.py::log_retcode] HackRF-SET_RF_GAIN to 0: Success
[INFO::Device.py::log_retcode] HackRF-SET_IF_GAIN to 16: Success
[INFO::Device.py::log_retcode] HackRF-SET_BB_GAIN to 14: Success
[INFO::Device.py::log_retcode] HackRF-SET_BIAS_TEE_ENABLED to 0: Success
[INFO::Device.py::log_retcode] HackRF-Start RX MODE: Success
[INFO::Device.py::stop_rx_mode] HackRF: Stopping RX Mode: Stopped receiving: Stop button clicked
[INFO::Device.py::log_retcode] HackRF-STOP RX MODE: Success
[INFO::Device.py::log_retcode] HackRF-CLOSE: Success
[INFO::Device.py::log_retcode] HackRF-EXIT: Success
[INFO::De

## Simple Replay Attack with URH
### 1. Spectrum Scanning
Having configured our SDR let’s start by scanning the wireless spectrum and find the center frequency of the device we want to attack. To open the spectrum analyzer use **File → Spectrum Analyzer**, this will give you a dialog as shown in the following figure. 
<img src="nbimages/URH_1_SpectrumAnalyzer.png" alt="Drawing" />
### 2. Record the Signal
To capture and replay a signal from a basic RF remote using URH, first connect a compatible SDR device (e.g., RTL-SDR or HackRF) and use the Record function in URH to capture the signal while pressing the button on the remote. Open up the recording dialog via **File → Record Signal**. 
<img src="nbimages/URH_2_Record.png" alt="Drawing" />
### 3. Open a Recording
You can open your capture file or the capture file under solutions folder `HighTek_A.cf32` to analyze the signal of interest. A signal can be added to Interpretation tab via **File → Open**. 
<img src="nbimages/URH_3_Interpretation.png" alt="Drawing" />
### 4. Send the Signal
If you're using a device like HackRF that supports transmission, you can use URH’s Transmit feature to replay the captured signal. This makes URH especially useful for reverse-engineering and testing RF devices. 

With the **triangle button** at the top of the following figure, you can replay the captured signal. Remember to set the frequency correctly! 

## Decoding with URH
### 1. Open a Recording
You can open your capture file or the capture file under solutions folder `HighTek_A.cf32` to analyze the signal of interest. A signal can be added to Interpretation tab via **File → Open**. 
<img src="nbimages/URH_decode_1_open.png" alt="Drawing" />
### 2. Set the Correct Sampling Rate
The `i` button opens signal detail dialog where you can also edit the sample rate. This influences the displayed time in Interpretation and Analysis. 
<img src="nbimages/URH_decode_2_samplerate.png" alt="Drawing" />
### 3. Demodulate
Change the Signal View from Analog to **Demodulated** and move the center between the areas with the mouse.
<img src="nbimages/URH_decode_3_demodulate.png" alt="Drawing" />
### 4. Configure Decoding
Once captured, you can analyze the waveform, inspect symbol patterns, and attempt to decode the underlying protocol. URH provides tools to help identify frame structures and bit patterns, even for unknown protocols.

Wireless protocols may use sophisticated encodings to prevent transmission errors or increase their energy efficiency. Therefore, URH comes with a powerful decoding component. You can access it via **Edit→ Decoding** from the main menu or by selecting **...** as decoding in the Analysis tab. This will open the decoding dialog shown below. Then save your new decoding with the Save as... button and it will be added to the list of available decodings in the Analysis tab.
<img src="nbimages/URH_decode_4_configure.png" alt="Drawing" />
### 5. Apply Decoding
To apply your crafted decoding to your data, just select the messages you want to set it for ( Ctrl + A for all) and use the **Configure Decoding combobox** in the Analysis tab.
<img src="nbimages/URH_decode_5_analysis.png" alt="Drawing" />


# VIII. Advanced Replay Attack with GNU Radio
What we have seen so far is **simple** capture and replay attack. It's pretty good considering how easy it is to accomplish. But keep in mind that we didn't have to decode this signal, we didn't have to even understand what the modulation was. Even if we don't know what the modulation is we could still capture a signal once we've identified it and then replay it and find out if that replay actually does anything. It's a pretty powerful tool to have in our toolbox being able to replay a signal. It's useful for testing systems to see if they respond to a replay packet and it's also extremely useful to create a as a means of creating a repeatable test vector that we can use. 

The **cleanest** way we could possibly transmit the signal would be actually decoding the information that was transmitted and **synthesizing** our new waveform that is generated on the CPU without actually capturing any noise at all.  

## 1. Demodulate
The first step is to **reverse the modulation scheme** that is used by the remote. We’ll use a pre-made GRC flowgraph to do this. Execute the following code cell to run it. 

- The GUI contains a **textbox to set the frequency**. **Set the frequency** to the correct value. You can use the spectrum display for this. 
- The GUI also contains a  **threshold slider**. **Gradually increase threshold** until you see a stable waveform appear. 

Then **measure** the timing on the waveform by clicking and dragging to zoom into one of the shortest transitions. Measure the timing in microseconds either by referring to the horizontal axis markings or by hovering the mouse over both sides of the short transition and calculating the time difference.

<img src="nbimages/grc_6_demodulate.png" alt="Drawing" /> 

There is a pre-made flowgraph, `6_Demodulate.grc` in the solutions directory that simply demodulates the recorded signal.  

Click the `...` to run the solution. 

In [8]:

!grcc solutions/6_Demod.grc 
!/usr/bin/python3 Demod.py --filename='keyfinder_red.dat'

bb60_iq.block.yml                        block: error: Value type 'NoneType' for 'inputs' not in valid types
sm_iq.block.yml                          block: error: Value type 'NoneType' for 'inputs' not in valid types
>>> Warning: vocoder_codec2_decode_ps - option_attributes are for enums only, ignoring
>>> Warning: vocoder_codec2_encode_sp - option_attributes are for enums only, ignoring
vsg60_iqin.block.yml                     block: error: Value type 'NoneType' for 'outputs' not in valid types
<<< Welcome to GNU Radio Companion Compiler 3.10.4.0 >>>

Block paths:
	/home/murat/.grc_gnuradio
	/usr/local/share/gnuradio/grc/blocks

>>> Loading: /home/murat/gitdir/EUGRDays2026/solutions/6_Demod.grc
>>> Generating: /home/murat/gitdir/EUGRDays2026/Demod.py


## 2. Decode
Once you set the frequency to **314.7MHz** and the threshold level exceeds roughly **100** (the exact level will depend on your local noise conditions) you should see a clear and stable digital waveform with a stable minimum transition time.

Next, you’ll use another GNU Radio Companion flowgraph to perform **clock recovery**, the process of extracting data bits from the demodulated waveform. We’ll use a pre-made GRC flowgraph to do this. Execute the following code cell to run it. 

Look closely at the notes in the flowgraph, set the following variable to the correct value for the flowgraph to function:
- **symbol_time** (the minimum timing measured in the previous step)

<img src="nbimages/grc_7_decode.png" alt="Drawing" /> 

There is a pre-made flowgraph, `7_Decode.grc` in the solutions directory that simply decodes the recorded signal.  

Click the `...` to run the solution. 

In [9]:

!grcc solutions/7_Decode.grc 
!/usr/bin/python3 Decode.py --filename='keyfinder_red.dat'

bb60_iq.block.yml                        block: error: Value type 'NoneType' for 'inputs' not in valid types
sm_iq.block.yml                          block: error: Value type 'NoneType' for 'inputs' not in valid types
>>> Warning: vocoder_codec2_decode_ps - option_attributes are for enums only, ignoring
>>> Warning: vocoder_codec2_encode_sp - option_attributes are for enums only, ignoring
vsg60_iqin.block.yml                     block: error: Value type 'NoneType' for 'outputs' not in valid types
<<< Welcome to GNU Radio Companion Compiler 3.10.4.0 >>>

Block paths:
	/home/murat/.grc_gnuradio
	/usr/local/share/gnuradio/grc/blocks

>>> Loading: /home/murat/gitdir/EUGRDays2026/solutions/7_Decode.grc
>>> Generating: /home/murat/gitdir/EUGRDays2026/Decode.py
correlate_access_code_tag_bb :debug: Access code: 8e88
correlate_access_code_tag_bb :debug: Mask: ffffff


## 3. Extract
You must set symbol_time to **315 microseconds**, as measured in Step 2. This variable is also used as a parameter in **SymbolSync** block and it is not runtime-configurable! So, this means you have to change the flowgraph at design time!

Open yet another flowgraph called `8_Extract.grc`. This flowgraph is similar to the previous one, but an additional block at the end of the signal chain: a ZMQ PUSH Message Sink. 

This block will take the payloads extracted from the flowgraph in the previous step and send them through a socket to a Python script for further processing. The Python script we’ve con gured is called rf_logger.py 

1. First, run the flowgraph. 
2. Next, run the Python script. 

Before quitting the script for the last time, it’s a good idea to kill the GNU Radio Companion execution window first, then hitting Ctrl-C in the terminal window running the script. Doing so in the opposite order will cause the GNU Radio window to hang.

<img src="nbimages/grc_8_extract.png" alt="Drawing" />

There is a pre-made flowgraph, `8_Extract.grc` in the solutions directory that simply extracts the bits.  

Click the `...` to run the solution. 

In [ ]:

!grcc solutions/8_Extract.grc 
!/usr/bin/python3 Extract.py --filename='keyfinder_red.dat'

bb60_iq.block.yml                        block: error: Value type 'NoneType' for 'inputs' not in valid types
sm_iq.block.yml                          block: error: Value type 'NoneType' for 'inputs' not in valid types
>>> Warning: vocoder_codec2_decode_ps - option_attributes are for enums only, ignoring
>>> Warning: vocoder_codec2_encode_sp - option_attributes are for enums only, ignoring
vsg60_iqin.block.yml                     block: error: Value type 'NoneType' for 'outputs' not in valid types
<<< Welcome to GNU Radio Companion Compiler 3.10.4.0 >>>

Block paths:
	/home/murat/.grc_gnuradio
	/usr/local/share/gnuradio/grc/blocks

>>> Loading: /home/murat/gitdir/EUGRDays2026/solutions/8_Extract.grc
>>> Generating: /home/murat/gitdir/EUGRDays2026/Extract.py
correlate_access_code_tag_bb :debug: Access code: e888
correlate_access_code_tag_bb :debug: Mask: fffffff
correlate_access_code_tag_bb :debug: writing tag at sample 4400
***** VERBOSE PDU DEBUG PRINT ******
()
pdu length =         80

## 4. Synthesize 
The last step is to create the desired signal on the fly, directly from **GNU Radio**. Below is the flowgraph that can be used to tranmit the bits we have obtained preivously over the air.  

<img src="nbimages/grc_9_synthesize.png" alt="Drawing" /> 

There is a pre-made flowgraph, `9_Synthesize.grc` in the solutions directory that simply synthesizes the signal from the bits.  

Click the `...` to run the solution. 

In [34]:

!grcc solutions/9_Synthesize.grc 
!/usr/bin/python3 Synthesize.py 

bb60_iq.block.yml                        block: error: Value type 'NoneType' for 'inputs' not in valid types
sm_iq.block.yml                          block: error: Value type 'NoneType' for 'inputs' not in valid types
>>> Warning: vocoder_codec2_decode_ps - option_attributes are for enums only, ignoring
>>> Warning: vocoder_codec2_encode_sp - option_attributes are for enums only, ignoring
vsg60_iqin.block.yml                     block: error: Value type 'NoneType' for 'outputs' not in valid types
<<< Welcome to GNU Radio Companion Compiler 3.10.4.0 >>>

Block paths:
	/home/murat/.grc_gnuradio
	/usr/local/share/gnuradio/grc/blocks

>>> Loading: /home/murat/gitdir/EUGRDays2026/solutions/9_Synthesize.grc
>>> Generating: /home/murat/gitdir/EUGRDays2026/Synthesize.py
gr-osmosdr 0.2.0.0 (0.2.0) gnuradio 3.10.4.0
built-in sink types: uhd hackrf bladerf soapy redpitaya file 
Using HackRF One with firmware 2024.02.1
UUUU

# IX. Son - The End
I hope you enjoyed the workshop and had pleasent time! 

Feel free to reach out to me at ytregitim at gmail dot com. 

Thanks for joining!